# 17. Two priorities over RCS and the 3GPP selection

Two flows on every vehicle, one urgent and one routine, scheduled either by the 3GPP sensing-based
selection or by Repeated Contention Scheduling (RCS, Rolich et al., MAIN 2026) with tones drawn
uniformly or skewed by the priority of the packet. `tools/rcs/priority_grid.py` ran the 192 cases on
ease-rtx, `tools/rcs/priority_metrics.py` measured them; `docs/rcs.md` describes the implementation.

| | periodic1 | aperiodic1 |
|---|---|---|
| traffic (groupcast-adaptation-ns3) | 100 ms, 300, 190, 190, 190, 190 B in turn | 50 ms + Exp(50 ms), 200 or 400 B |
| QoS | PQI 90, SL priority 5, PDB 100 ms | PQI 89, SL priority 2, PDB 50 ms |
| who, when | every vehicle, the whole run | 70 % of the vehicles, picked at random, from 10 to 20 s |
| 3GPP scheduling | SB-SPS, RRI 100 ms | dynamic, one transmission (or two: `_retx1`) |

aperiodic1's sizes are capped at 400 B: an RCS winner sends on one sub-channel of 25 RBs, which
carries 509 B of payload at MCS 13. Every scheme of one seed carries exactly the same packets.

* **Radio**: mu = 1, 20 MHz, sub-channels of 25 RBs, MCS 13, 30 s, the first 2 s left out.
* **Load**: n = 108, 180, 252 vehicles, so periodic1 alone is a = 0.3, 0.5, 0.7 of RCS's 36 SCs per
  10 ms (the paper's load). aperiodic1 adds about 0.7 a during the burst.
* **square**: the paper's model, static vehicles in a 10 m square, all-or-nothing reception, every
  UE hears every tone. **highway**: 2 km, 3 lanes per direction (SUMO), the PHY decides, pairs
  within 300 m; RCS tones heard by all or within 600, 300, 150 m.
* **3GPP**: re-evaluation and pre-emption off (`3gpp_off`), re-evaluation (`3gpp_reeval`), both
  (`3gpp_reeval_preempt`, the baseline); each also with one blind retransmission of aperiodic1
  (`_retx1`). **RCS**: J = 1 and 36 groups, `uniform` or `skewed` tones.
* **Phases**: `before` (2 to 10 s), `burst` (10 to 20 s), `after` (20 to 29.3 s), `all`.
* **Metrics** (paper): PRR, Pcoll, Pdrop (never sent within the PDB), E[PIR], E[AoI], V(Ath).
  Values are the mean of 2 seeds.

In [ ]:
import os, sys, json, subprocess
sys.path.insert(0, os.path.abspath("../../tools"))
sys.path.insert(0, os.path.abspath("../../tools/qos"))
import morev2x_results as mr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

mr.apply_chart_style()
os.chdir(mr.ROOT)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

DATA = "results/rcs_priority/"
runs = pd.read_csv(DATA + "summary.csv")
runs["config"] = np.where(runs.scheme == "rcs",
                          "rcs_J" + runs.groups.fillna(0).astype(int).astype(str) + "_" + runs.choice.fillna("") + "_" + runs.tone.fillna(""),
                          runs.scheme)
def table(name):
    return pd.read_csv(DATA + name + ".csv").merge(runs, left_on="run", right_on="name")
paper, reevals, rcs, cbr = table("paper"), table("reevaluations"), table("rcs"), table("cbr_time")
def mean(frame, rows, values, columns="load"):
    return frame.groupby(rows + [columns])[values].mean().unstack(columns)
runs.groupby(["model", "scheme"]).size().unstack(0)

## 1. The burst raises the channel load

Mean CBR over the UEs every 100 ms, for the 3GPP baseline and RCS (J = 36, skewed). The grey band
is the aperiodic1 burst.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, model in zip(axes, ["square", "highway"]):
    for (config, style), load in [((c, s), l) for c, s in (("3gpp_reeval_preempt", "-"), ("rcs_J36_skewed_all", "--"))
                                  for l in (0.3, 0.5, 0.7)]:
        part = cbr[(cbr.model == model) & (cbr.config == config) & (cbr.load == load)]
        series = part.groupby("time")["mean"].mean()
        ax.plot(series.index, series.values, style, label="%s a=%.1f" % (config.replace("_all", ""), load),
                color=mr.SERIES_COLORS[[0.3, 0.5, 0.7].index(load)])
    ax.axvspan(10, 20, color="0.85", zorder=0)
    ax.set_title(model)
    ax.set_xlabel("time (s)")
axes[0].set_ylabel("mean CBR")
axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()
mean(cbr.assign(phase=pd.cut(cbr.time, [0, 10, 20, 31], labels=["before", "burst", "after"], right=False))
     .query("config == '3gpp_reeval_preempt'"), ["model", "phase"], "mean").round(3)

With the 3GPP baseline the CBR rises during the burst and falls back after it: by 0.15, 0.21 and
0.22 in the square (0.59 to 0.81 at a = 0.7) and by 0.09, 0.15 and 0.20 on the highway (0.27 to 0.47
at a = 0.7). In the square every UE senses every transmission, so its CBR is the share of the whole
channel in use.

## 2. The paper's model (square)

### PRR during the burst, per flow and for both flows

In [ ]:
burst = paper[paper.phase == "burst"]
square = burst[burst.model == "square"]
mean(square, ["config", "flow"], "prr").round(3)

In [ ]:
configs = ["3gpp_reeval_preempt", "3gpp_reeval_preempt_retx1", "rcs_J1_uniform_all", "rcs_J1_skewed_all",
           "rcs_J36_uniform_all", "rcs_J36_skewed_all"]
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True)
for row, metric, label in ((0, "prr", "PRR"), (1, "aoi_ms", "E[AoI] (ms)")):
    for col, flow in enumerate(["aperiodic1", "periodic1", "all"]):
        ax = axes[row, col]
        for i, config in enumerate(configs):
            part = square[(square.flow == flow) & (square.config == config)].groupby("load")[metric].mean()
            ax.plot(part.index, part.values, marker="os^v<>"[i], label=config.replace("_all", ""),
                    linestyle="-" if "rcs" in config else "--")
        ax.set_title("%s, %s" % (flow, label))
        ax.grid(alpha=0.3)
        if row == 1:
            ax.set_xlabel("load a (periodic1 alone)")
            ax.set_yscale("log")
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
cols = ["prr", "pcoll", "pdrop", "pir_ms", "aoi_ms", "aoi_violation_100", "aoi_violation_200", "aoi_violation_500"]
square[square.config.isin(configs) & (square.load == 0.7)].groupby(["config", "flow"])[cols].mean().round(3)

* **The urgent flow.** RCS with skewed tones and J = 36 keeps aperiodic1 at PRR 0.99, 0.98 and 0.96
  for a = 0.3, 0.5, 0.7, with E[AoI] at 63 to 66 ms. The 3GPP baseline gives 0.75, 0.56 and 0.35
  (0.92, 0.70 and 0.38 with the retransmission), and its AoI reaches 220 to 240 ms at a = 0.7.
* **Both flows together.** Uniform RCS with J = 36 is best: PRR 0.99, 0.96 and 0.80, against 0.87,
  0.76 and 0.62 for the 3GPP baseline.
* **Losses.** The 3GPP losses are collisions (Pdrop is 0: the selection always finds a resource).
  RCS loses aperiodic1 to drops at high load with uniform tones (no win within the 50 ms PDB),
  and periodic1 to drops and collisions when the skew pushes it back.

### What the skew buys and what it costs

Skewed minus uniform, per flow, phase and J (square):

In [ ]:
rcs_paper = paper[(paper.scheme == "rcs") & (paper.model == "square")]
gain = (rcs_paper.groupby(["groups", "choice", "flow", "phase", "load"]).prr.mean().unstack("choice"))
gain = (gain.skewed - gain.uniform).unstack("load").round(3)
gain.loc[(slice(None), slice(None), ["burst", "before"]), :]

* **During the burst** the skew moves PRR from periodic1 to aperiodic1. With J = 36: aperiodic1
  +0.00, +0.05, +0.24 and periodic1 -0.04, -0.10, -0.24 at a = 0.3, 0.5, 0.7. With J = 1 the gain
  for aperiodic1 only shows at a = 0.7 (+0.13).
* **The skew also costs periodic1 when it is alone** (before the burst): -0.03 to -0.10 with
  J = 36, -0.07 to -0.13 with J = 1. Concentrating every periodic1 tone in the upper part of the
  group makes equal tones, hence collisions, more likely. With J = 36 at mu = 1 a group has only
  m = 5 tones.
* **Both flows together** lose PRR with the skew. It is a priority mechanism: the urgent flow gains,
  at more than its cost for the routine flow.

## 3. Re-evaluation and pre-emption (3GPP)

Checks and resource changes per flow during the burst. In the `off` runs MoReV2X still runs the
check and counts the changes it would have made, without applying them.

In [ ]:
r = reevals[(reevals.phase == "burst") & reevals.scheme.str.startswith("3gpp")]
mean(r, ["model", "scheme", "flow", "mechanism"], ["checks", "changes"]).round(0)

Contribution of each mechanism to the PRR during the burst: re-evaluation is `3gpp_reeval` minus
`3gpp_off`, pre-emption `3gpp_reeval_preempt` minus `3gpp_reeval`, with and without the
retransmission of aperiodic1.

In [ ]:
g = paper[(paper.phase == "burst") & paper.scheme.str.startswith("3gpp")]
g = g.groupby(["model", "scheme", "flow", "load"]).prr.mean().unstack("scheme")
rows = {}
for suffix in ("", "_retx1"):
    rows["reeval" + suffix] = g["3gpp_reeval" + suffix] - g["3gpp_off" + suffix]
    rows["preempt" + suffix] = g["3gpp_reeval_preempt" + suffix] - g["3gpp_reeval" + suffix]
    rows["retx" + suffix] = g["3gpp_reeval_preempt" + suffix] - g["3gpp_reeval_preempt"] if suffix else None
contribution = pd.DataFrame({k: v for k, v in rows.items() if v is not None}).unstack("load").round(3)
contribution

* **Pre-emption needs an announced reservation.** With aperiodic1 dynamic and sent once, it never
  announces a future resource, so pre-emption checks periodic1's reservations about 9,000 to
  23,000 times per burst and changes none: `3gpp_reeval_preempt` equals `3gpp_reeval` exactly.
  With the retransmission announced in the SCI, periodic1 gives up 20 to 1,000 resources per burst,
  but PRR moves by at most about 0.01.
* **Re-evaluation** matters once aperiodic1 announces its retransmission: it changes 1,500 to
  8,700 of aperiodic1's resources per burst in the square and lifts its PRR by about 0.08, 0.12 and
  0.07 (a = 0.3, 0.5, 0.7). Without the retransmission it changes almost nothing (a one-shot packet
  has nothing ahead of it to re-check).
* **The retransmission itself** helps aperiodic1 (+0.17, +0.14, +0.02 of PRR in the square at
  a = 0.3, 0.5, 0.7), but its extra transmissions cost periodic1 (-0.00, -0.07, -0.16), so both
  flows together lose 0.09 at a = 0.7.

## 4. The highway, and who hears the tones

In [ ]:
highway = burst[burst.model == "highway"]
keep = ["3gpp_reeval_preempt", "3gpp_reeval_preempt_retx1"] + sorted(c for c in highway.config.unique() if c.startswith("rcs"))
mean(highway[highway.config.isin(keep)], ["config", "flow"], "prr").round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
tones = ["all", "range600", "range300", "range150"]
for ax, flow in zip(axes, ["aperiodic1", "periodic1"]):
    for j, marker in ((1, "o"), (36, "s")):
        for choice, style in (("uniform", "--"), ("skewed", "-")):
            values = [highway[(highway.flow == flow) & (highway.groups == j) & (highway.choice == choice)
                              & (highway.tone == t) & (highway.load == 0.7)].prr.mean() for t in tones]
            ax.plot(tones, values, style, marker=marker, label="RCS J=%d %s" % (j, choice))
    base = highway[(highway.flow == flow) & (highway.config == "3gpp_reeval_preempt") & (highway.load == 0.7)].prr.mean()
    ax.axhline(base, color="k", linewidth=1, label="3GPP baseline")
    ax.set_title("%s, a = 0.7, within 300 m" % flow)
    ax.set_xlabel("tones heard")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("PRR (burst)")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

* **Tones heard by all** on 2 km behave as in the square, but the whole road contends for the same
  36 SCs: at a = 0.7 RCS falls below the 3GPP baseline (both flows 0.78 to 0.85 against 0.885),
  while the skew still protects aperiodic1 (0.96 with J = 36).
* **Tones heard within 600 m** give the best result on the highway: J = 36 reaches 0.94 for both
  flows at a = 0.7, above the 3GPP baseline (0.885), since UEs far apart reuse the same SCs.
* **Shorter ranges** bring hidden terminals: two winners that cannot hear each other take the same
  SC. J = 36 still holds 0.84 at 150 m; J = 1 falls to 0.62 at 300 m and 0.42 at 150 m, as in
  notebook 07.
* **The skew stops working when few tones are heard.** With a limited range a UE competes with its
  neighbours only, most contenders win, and the losses are hidden-terminal collisions that the
  tone choice cannot avoid: skewed and uniform stay within about 0.05 of each other per flow.

## 5. Inside RCS (square, burst)

In [ ]:
inner = rcs[(rcs.phase == "burst") & (rcs.model == "square")]
mean(inner, ["groups", "choice", "flow"], ["contentions_per_packet", "access_delay_ms", "yielded",
                                          "dropped_no_win", "dropped_waiting"]).round(2)

* `yielded`: a periodic1 packet gave the UE's contention to an aperiodic1 packet of the same UE
  before the frame started (outcome 4); `dropped_waiting`: dropped while the other flow held it.
* With the skew, aperiodic1 needs fewer contentions and less access delay, periodic1 more: at
  a = 0.7 and J = 36, 1.35 against 2.92 contentions per aperiodic1 packet, 3.57 against 2.89 per
  periodic1 packet.

## 6. Two seeds

The spread between the two seeds, per metric and configuration, during the burst:

In [ ]:
spread = burst.groupby(["model", "config", "flow", "load"]).prr.agg(lambda x: x.max() - x.min())
print("median spread %.3f, 95th percentile %.3f, largest %.3f at %s" % (spread.median(), spread.quantile(0.95), spread.max(), spread.idxmax()))

Most configurations agree within a few hundredths. The largest spreads are RCS with J = 1 at
a = 0.7, where one contention per frame with 36 winners sits at the edge of its capacity and small
differences in the arrivals decide how many packets miss their PDB.

## Summary

* For the urgent flow, RCS with priority-skewed tones and J = 36 in the paper's model is far ahead
  of the 3GPP selection: PRR 0.96 at a = 0.7 during the burst against 0.35 to 0.38, AoI 66 ms
  against 220 to 240 ms.
* The skew is a priority lever, not a free gain: aperiodic1 gains up to +0.24 of PRR, periodic1
  loses as much, and the skew costs periodic1 even with no urgent traffic.
* 3GPP pre-emption does nothing for dynamic one-shot urgent traffic, which announces no
  reservation; re-evaluation helps once a retransmission is announced (up to +0.12 of PRR).
* On the highway the gains need the tones to be heard over a few hundred metres: within 600 m RCS
  J = 36 is the best of all; with tones heard by all it loses its spatial reuse, and with short
  ranges the skew has nothing left to decide.

To regenerate: `tools/rcs/priority_grid.py --summary` on the grid's results writes `summary.csv`
and the stacked tables into `results/rcs_priority/` (not in git); then run this notebook.